In [2]:
# Install required libraries
!pip install pandas numpy scikit-learn torch

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import numpy as np

path = "/content/drive/MyDrive/BigData/assignment3/ml-latest-small/"

ratings = pd.read_csv(path + "ratings.csv")
movies = pd.read_csv(path + "movies.csv")

# -----------------------------
# Extract year from title
# -----------------------------
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['year'] = pd.to_numeric(movies['year'], errors='coerce')
movies['year'] = movies['year'].fillna(movies['year'].median())

# -----------------------------
# One-hot encode genres
# -----------------------------
movies['genres'] = movies['genres'].str.split('|')
all_genres = list(set(g for sublist in movies['genres'] for g in sublist))

for g in all_genres:
    movies[g] = movies['genres'].apply(lambda x: int(g in x))

# -----------------------------
# Movie average rating
# -----------------------------
movie_avg = ratings.groupby('movieId')['rating'].mean().reset_index()
movie_avg.rename(columns={'rating': 'avg_rating'}, inplace=True)

movies = movies.merge(movie_avg, on='movieId', how='left')
movies['avg_rating'] = movies['avg_rating'].fillna(movies['avg_rating'].mean())

# -----------------------------
# Normalize year and avg_rating
# -----------------------------
movies['year'] = (movies['year'] - movies['year'].min()) / (movies['year'].max() - movies['year'].min())
movies['avg_rating'] = (movies['avg_rating'] - movies['avg_rating'].min()) / (movies['avg_rating'].max() - movies['avg_rating'].min())

In [7]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
movies.head()

,movieId,title,genres,year,Film-Noir,Children,Western,Comedy,Crime,Mystery,...,Adventure,Action,Animation,Romance,Musical,Thriller,IMAX,Fantasy,Horror,avg_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",0.801724,0,1,0,1,0,0,...,1,0,1,0,0,0,0,1,0,0.760207
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",0.801724,0,1,0,0,0,0,...,1,0,0,0,0,0,0,1,0,0.651515
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",0.801724,0,0,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0.613248
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",0.801724,0,0,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0.412698
4,5,Father of the Bride Part II (1995),[Comedy],0.801724,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0.571429


In [5]:
# Merge ratings with movie genres
data = ratings.merge(movies[['movieId'] + all_genres], on='movieId')

# Compute user preference per genre
user_genre_pref = {}

for user, group in data.groupby('userId'):
    genre_scores = {}
    for g in all_genres:
        if group[g].sum() > 0:
            genre_scores[g] = (group['rating'] * group[g]).sum() / group[g].sum()
        else:
            genre_scores[g] = 0
    user_genre_pref[user] = genre_scores

user_features = pd.DataFrame.from_dict(user_genre_pref, orient='index')
user_features = user_features.fillna(0)

In [6]:
user_features.head()

,Film-Noir,Children,Western,Comedy,Crime,Mystery,Sci-Fi,Drama,Documentary,(no genres listed),War,Adventure,Action,Animation,Romance,Musical,Thriller,IMAX,Fantasy,Horror
1,5.0,4.547619,4.285714,4.277108,4.355556,4.166667,4.225000,4.529412,0.000000,0.0,4.500000,4.388235,4.322222,4.689655,4.307692,4.681818,4.145455,0.000000,4.297872,3.470588
2,0.0,0.000000,3.500000,4.000000,3.800000,4.000000,3.875000,3.882353,4.333333,0.0,4.500000,4.166667,3.954545,0.000000,4.500000,0.000000,3.700000,3.750000,0.000000,3.000000
3,0.0,0.500000,0.000000,1.000000,0.500000,5.000000,4.200000,0.750000,0.000000,0.0,0.500000,2.727273,3.571429,0.500000,0.500000,0.500000,4.142857,0.000000,3.375000,4.687500
4,4.0,3.800000,3.800000,3.509615,3.814815,3.478261,2.833333,3.483333,4.000000,0.0,3.571429,3.655172,3.320000,4.000000,3.379310,4.000000,3.552632,3.000000,3.684211,4.250000
5,0.0,4.111111,3.000000,3.466667,3.833333,4.000000,2.500000,3.800000,0.000000,0.0,3.333333,3.250000,3.111111,4.333333,3.090909,4.400000,3.555556,3.666667,4.142857,3.000000


In [8]:
movie_features = movies[['movieId', 'year', 'avg_rating'] + all_genres]
movie_features = movie_features.set_index('movieId')

In [9]:
movie_features.head()

,year,avg_rating,Film-Noir,Children,Western,Comedy,Crime,Mystery,Sci-Fi,Drama,...,War,Adventure,Action,Animation,Romance,Musical,Thriller,IMAX,Fantasy,Horror
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.801724,0.760207,0,1,0,1,0,0,0,0,...,0,1,0,1,0,0,0,0,1,0
2,0.801724,0.651515,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
3,0.801724,0.613248,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0.801724,0.412698,0,0,0,1,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
5,0.801724,0.571429,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
import torch
from torch.utils.data import Dataset

class MovieLensDataset(Dataset):
    def __init__(self, ratings, user_features, movie_features):
        self.ratings = ratings
        self.user_features = user_features
        self.movie_features = movie_features

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        row = self.ratings.iloc[idx]
        user = row['userId']
        movie = row['movieId']
        rating = row['rating']

        user_vec = torch.tensor(self.user_features.loc[user].values, dtype=torch.float32)
        movie_vec = torch.tensor(self.movie_features.loc[movie].values, dtype=torch.float32)

        return user_vec, movie_vec, torch.tensor(rating, dtype=torch.float32)

dataset = MovieLensDataset(ratings, user_features, movie_features)

In [11]:
from torch.utils.data import random_split, DataLoader

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256)

In [12]:
import torch.nn as nn

class RecommenderNet(nn.Module):
    def __init__(self, user_dim, movie_dim, embed_dim=32):
        super().__init__()

        # User branch
        self.user_net = nn.Sequential(
            nn.Linear(user_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embed_dim)
        )

        # Movie branch
        self.movie_net = nn.Sequential(
            nn.Linear(movie_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embed_dim)
        )

        # Final prediction
        self.fc = nn.Sequential(
            nn.Linear(embed_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, user, movie):
        user_emb = self.user_net(user)
        movie_emb = self.movie_net(movie)

        combined = torch.cat([user_emb, movie_emb], dim=1)
        out = self.fc(combined)

        return out.squeeze()

In [13]:
user_dim = user_features.shape[1]
movie_dim = movie_features.shape[1]

model = RecommenderNet(user_dim, movie_dim)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

RecommenderNet(
  (user_net): Sequential(
    (0): Linear(in_features=20, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
  (movie_net): Sequential(
    (0): Linear(in_features=22, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
  (fc): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [14]:
import torch.optim as optim

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for user, movie, rating in train_loader:
        user, movie, rating = user.to(device), movie.to(device), rating.to(device)

        optimizer.zero_grad()
        pred = model(user, movie)
        loss = criterion(pred, rating)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 1.1263
Epoch 2, Loss: 0.6644
Epoch 3, Loss: 0.6586
Epoch 4, Loss: 0.6537
Epoch 5, Loss: 0.6560
Epoch 6, Loss: 0.6535
Epoch 7, Loss: 0.6512
Epoch 8, Loss: 0.6483
Epoch 9, Loss: 0.6504
Epoch 10, Loss: 0.6514


In [15]:
from sklearn.metrics import mean_squared_error

model.eval()
preds, actuals = [], []

with torch.no_grad():
    for user, movie, rating in test_loader:
        user, movie = user.to(device), movie.to(device)
        pred = model(user, movie)

        preds.extend(pred.cpu().numpy())
        actuals.extend(rating.numpy())

rmse = np.sqrt(mean_squared_error(actuals, preds))
print("Neural Network RMSE:", rmse)

Neural Network RMSE: 0.8096823034898055


In [17]:
def recommend_nn(user_id, model, user_features, movie_features, movies_df, top_k=10):
    model.eval()

    user_vec = torch.tensor(user_features.loc[user_id].values, dtype=torch.float32).unsqueeze(0).to(device)

    scores = []

    with torch.no_grad():
        for movie_id in movie_features.index:
            movie_vec = torch.tensor(movie_features.loc[movie_id].values, dtype=torch.float32).unsqueeze(0).to(device)

            pred_rating = model(user_vec, movie_vec).item()
            scores.append((movie_id, pred_rating))

    # Sort by predicted rating
    scores.sort(key=lambda x: x[1], reverse=True)

    top_movies = scores[:top_k]

    # Get movie titles
    result = []
    for movie_id, score in top_movies:
        title = movies_df[movies_df['movieId'] == movie_id]['title'].values[0]
        result.append((title, round(score, 3)))

    return result

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_cosine(user_id, ratings, movies, movie_features, top_k=10):

    # Get movies rated by user
    user_data = ratings[ratings['userId'] == user_id]

    if len(user_data) == 0:
        return []

    # Build user profile (weighted genre vector)
    genre_matrix = movie_features.loc[user_data['movieId']][all_genres].values
    weights = user_data['rating'].values.reshape(-1, 1)

    user_profile = np.sum(genre_matrix * weights, axis=0)
    user_profile = user_profile.reshape(1, -1)

    # Compute similarity with all movies
    all_movie_genres = movie_features[all_genres].values
    similarity = cosine_similarity(user_profile, all_movie_genres)[0]

    # Rank movies
    movie_ids = movie_features.index.tolist()
    scores = list(zip(movie_ids, similarity))
    scores.sort(key=lambda x: x[1], reverse=True)

    top_movies = scores[:top_k]

    result = []
    for movie_id, score in top_movies:
        title = movies[movies['movieId'] == movie_id]['title'].values[0]
        result.append((title, round(score, 3)))

    return result

In [19]:
user_id = 1

print("🔵 Neural Network Recommendations:")
nn_recs = recommend_nn(user_id, model, user_features, movie_features, movies)
for i, (title, score) in enumerate(nn_recs, 1):
    print(f"{i}. {title} (Predicted Rating: {score})")

print("\n🟢 Cosine Similarity Recommendations:")
cos_recs = recommend_cosine(user_id, ratings, movies, movie_features)
for i, (title, score) in enumerate(cos_recs, 1):
    print(f"{i}. {title} (Similarity: {score})")

🔵 Neural Network Recommendations:
1. My Life as McDull (Mak dau goo si) (2001) (Predicted Rating: 5.62)
2. Alesha Popovich and Tugarin the Dragon (2004) (Predicted Rating: 5.615)
3. Runaway Brain (1995)  (Predicted Rating: 5.604)
4. Priklyucheniya Kapitana Vrungelya (1979) (Predicted Rating: 5.574)
5. Dream of Light (a.k.a. Quince Tree Sun, The) (Sol del membrillo, El) (1992) (Predicted Rating: 5.573)
6. Happy Feet Two (2011) (Predicted Rating: 5.571)
7. On the Ropes (1999) (Predicted Rating: 5.567)
8. Bobik Visiting Barbos (1977) (Predicted Rating: 5.561)
9. Scooby-Doo Goes Hollywood (1979) (Predicted Rating: 5.559)
10. More (1998) (Predicted Rating: 5.555)

🟢 Cosine Similarity Recommendations:
1. Dragonheart 2: A New Beginning (2000) (Similarity: 0.864)
2. Hunting Party, The (2007) (Similarity: 0.843)
3. Flashback (1990) (Similarity: 0.827)
4. The Great Train Robbery (1978) (Similarity: 0.827)
5. Stunt Man, The (1980) (Similarity: 0.822)
6. Extreme Days (2001) (Similarity: 0.812)
7. 

In [21]:
def precision_recall_nn(model, test_loader, k=10, threshold=4):
    model.eval()

    user_preds = {}

    with torch.no_grad():
        for user, movie, rating in test_loader:
            user, movie = user.to(device), movie.to(device)
            preds = model(user, movie)

            for i in range(len(user)):
                uid = tuple(user[i].cpu().numpy())  # unique user vector
                if uid not in user_preds:
                    user_preds[uid] = []

                user_preds[uid].append((preds[i].item(), rating[i].item()))

    precisions, recalls = [], []

    for uid, values in user_preds.items():
        # sort by predicted rating
        values.sort(key=lambda x: x[0], reverse=True)
        top_k = values[:k]

        n_rel = sum((true >= threshold) for (_, true) in values)
        n_rec_k = sum((pred >= threshold) for (pred, _) in top_k)
        n_rel_and_rec_k = sum((true >= threshold and pred >= threshold) for (pred, true) in top_k)

        precision = n_rel_and_rec_k / n_rec_k if n_rec_k > 0 else 0
        recall = n_rel_and_rec_k / n_rel if n_rel > 0 else 0

        precisions.append(precision)
        recalls.append(recall)

    return np.mean(precisions), np.mean(recalls)

p_nn, r_nn = precision_recall_nn(model, test_loader, k=10)
print(f"NN Precision@10: {p_nn:.4f}")
print(f"NN Recall@10: {r_nn:.4f}")

NN Precision@10: 0.6237
NN Recall@10: 0.3477


In [22]:
def precision_recall_cosine(user_features, movie_features, ratings, k=10, threshold=4):
    from sklearn.metrics.pairwise import cosine_similarity

    precisions, recalls = [], []

    for user_id in ratings['userId'].unique():
        user_data = ratings[ratings['userId'] == user_id]

        if len(user_data) < 5:
            continue

        # Build user profile
        genre_matrix = movie_features.loc[user_data['movieId']][all_genres].values
        weights = user_data['rating'].values.reshape(-1, 1)
        user_profile = np.sum(genre_matrix * weights, axis=0).reshape(1, -1)

        # Similarity
        all_movie_genres = movie_features[all_genres].values
        sim = cosine_similarity(user_profile, all_movie_genres)[0]

        movie_ids = movie_features.index.tolist()
        scores = list(zip(movie_ids, sim))
        scores.sort(key=lambda x: x[1], reverse=True)

        top_k_movies = [m for m, _ in scores[:k]]

        # Ground truth relevant movies
        relevant = set(user_data[user_data['rating'] >= threshold]['movieId'])
        recommended = set(top_k_movies)

        n_rel = len(relevant)
        n_rec_k = len(recommended)
        n_rel_and_rec_k = len(relevant & recommended)

        precision = n_rel_and_rec_k / n_rec_k if n_rec_k > 0 else 0
        recall = n_rel_and_rec_k / n_rel if n_rel > 0 else 0

        precisions.append(precision)
        recalls.append(recall)

    return np.mean(precisions), np.mean(recalls)

p_cos, r_cos = precision_recall_cosine(user_features, movie_features, ratings)
print(f"Cosine Precision@10: {p_cos:.4f}")
print(f"Cosine Recall@10: {r_cos:.4f}")

Cosine Precision@10: 0.0395
Cosine Recall@10: 0.0100
